# Phase 3 RQ1 v2 — prepare inputs

This CPU-capable step freezes the ten-language XNLI sample selection and prepares URIEL+ and sentence-embedding priors. Turn Kaggle Internet on. A GPU is optional and only speeds up embedding inference; no model is fine-tuned here. A Hugging Face token is not required for the public XNLI data or embedding model.

The output ZIP contains the protocol, prior matrices, and data hash—not XNLI sentence text. Keep it in a private Kaggle dataset and attach it to the training and analysis notebooks. Grambank is excluded when its coverage report shows missing languages. The published-interference prior is excluded unless you explicitly provide a verified numeric 10x10 `interference_matrix.csv`.

Before running: push the repository changes to GitHub so this notebook can clone them.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Mani-S-49/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
elif (REPO_ROOT / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
else:
    raise FileExistsError(f"Repository path exists but is not a Git clone: {REPO_ROOT}")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "datasets==5.0.1", "urielplus==1.3.3", "sentence-transformers",
], check=True)
sys.path.insert(0, str(REPO_ROOT))

OUTPUT_DIR = Path("/kaggle/working/phase3_rq1_v2_inputs")
command = [
    sys.executable, "-m", "phase3.rq1_v2_workflow", "prepare-inputs",
    "--protocol", str(REPO_ROOT / "phase3" / "configs" / "rq1_v2_10lang_protocol.json"),
    "--output-dir", str(OUTPUT_DIR),
]
interference_matrices = list(Path("/kaggle/input").rglob("interference_matrix.csv"))
if len(interference_matrices) > 1:
    raise ValueError("Attach at most one interference_matrix.csv.")
if interference_matrices:
    command.extend(["--interference-matrix", str(interference_matrices[0])])
    print("Validating the supplied interference matrix for complete 10-language coverage.")
else:
    print("No numeric 10x10 interference matrix attached; that prior will be excluded.")
subprocess.run(command, cwd=REPO_ROOT, check=True)

INPUT_ZIP = OUTPUT_DIR / "phase3_rq1_v2_inputs.zip"
if not INPUT_ZIP.is_file():
    raise FileNotFoundError(f"Input bundle was not created: {INPUT_ZIP}")
print(f"Create a private Kaggle dataset from: {INPUT_ZIP}")